# Clase 121 — Custom training loops

Escribimos un **training loop manual** en TF con `GradientTape`: control total
sobre cada paso (útil para GANs, RL, multi-optimizer, debugging). El patrón es
`tape.gradient(loss, vars)` → `optimizer.apply_gradients(zip(grads, vars))`.

Requiere: `tensorflow` / `keras` (≥ 3.0).

## 1. Datos y modelo (MLP sobre Fashion-MNIST vía `tf.data`)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
            .shuffle(1024).batch(128).prefetch(tf.data.AUTOTUNE))

modelo = keras.Sequential([
    keras.Input((784,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10),                       # logits (sin softmax)
])
print("batches por época:", len(train_ds))

## 2. Loss, optimizer y métricas manuales

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
train_acc = keras.metrics.SparseCategoricalAccuracy()
train_loss = keras.metrics.Mean()
print("loss, optimizer y métricas listos")

## 3. Un `train_step` con `GradientTape`

In [ ]:
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = modelo(x, training=True)
        loss = loss_fn(y, logits)
    grads = tape.gradient(loss, modelo.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo.trainable_variables))
    train_loss.update_state(loss)
    train_acc.update_state(y, logits)
    return loss

xb, yb = next(iter(train_ds))
print("loss de un batch:", float(train_step(xb, yb)))

## 4. Loop completo por épocas y batches

In [ ]:
EPOCAS = 3
for epoca in range(EPOCAS):
    train_loss.reset_state()
    train_acc.reset_state()
    for xb, yb in train_ds:
        train_step(xb, yb)
    print(f"época {epoca + 1}: loss={train_loss.result():.4f} "
          f"acc={train_acc.result():.4f}")

## 5. `tape.watch` para tensores que no son `tf.Variable`

In [ ]:
x = tf.constant(3.0)                  # constant, no Variable -> hay que watch
with tf.GradientTape() as tape:
    tape.watch(x)
    y = x ** 3                       # dy/dx = 3x^2 = 27 en x=3
print("dy/dx con tape.watch:", float(tape.gradient(y, x)), "(esperado 27)")

## 6. Acelerar el step con `@tf.function`

In [ ]:
@tf.function
def train_step_rapido(x, y):
    with tf.GradientTape() as tape:
        logits = modelo(x, training=True)
        loss = loss_fn(y, logits)
    grads = tape.gradient(loss, modelo.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo.trainable_variables))
    return loss

xb, yb = next(iter(train_ds))
print("loss (step compilado):", float(train_step_rapido(xb, yb)))
print("El mismo loop con @tf.function corre 2-10x más rápido.")

## Ejercicios

1. **TF loop manual**: entrená un MLP en Fashion-MNIST con `GradientTape` y
   logging manual de loss y accuracy por época.
2. **PyTorch equivalente**: reimplementá el mismo loop con
   `optimizer.zero_grad(); loss.backward(); optimizer.step()`.
3. **Lightning**: resolvé el mismo problema con un `LightningModule.training_step`.
4. **Speedup con jit**: envolvé el `train_step` con `@tf.function` (TF) y con
   `torch.compile` (PyTorch); medí la mejora.
5. **Multi-optimizer**: escribí un loop que aplique un optimizer con LR bajo a
   las capas viejas y otro con LR alto a las nuevas.

## Conclusiones

- El patrón manual: `with tf.GradientTape() as tape:` → `tape.gradient(loss, vars)` → `optimizer.apply_gradients(zip(grads, vars))`.
- Las métricas se manejan a mano: `update_state` en cada batch y `reset_state` al inicio de cada época.
- `tape.watch(t)` fuerza a registrar tensores que no son `tf.Variable` (constantes).
- En capas con `BatchNorm`/`Dropout` hay que pasar `training=True` explícitamente en el forward.
- Envolver el `train_step` en `@tf.function` da el speedup del grafo sin perder el control del loop.

## ✅ Soluciones de los ejercicios

Custom training loops en TF (`GradientTape`), PyTorch y Lightning. Se validan por AST (no hay TF ni torch instalados). Incluyen el mismo problema en los tres frameworks, aceleración con `tf.function`/`torch.compile` y un loop multi-optimizer (discriminative learning rates).

**Ej. 1 — TF loop manual.** Entrenar con `GradientTape`, con logging de loss y accuracy.

In [ ]:
import tensorflow as tf
from tensorflow import keras

(x_train, y_train), _ = keras.datasets.fashion_mnist.load_data()
x_train = x_train.reshape(-1, 784) / 255.
ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(1024).batch(64)

model = keras.Sequential([keras.Input((784,)), keras.layers.Dense(128, activation="relu"),
                          keras.layers.Dense(10)])
opt = keras.optimizers.Adam(1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
acc = keras.metrics.SparseCategoricalAccuracy()

for step, (xb, yb) in enumerate(ds):
    with tf.GradientTape() as tape:
        logits = model(xb, training=True)
        loss = loss_fn(yb, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    acc.update_state(yb, logits)
    if step % 200 == 0:
        print(f"step {step} | loss {float(loss):.3f} | acc {float(acc.result()):.3f}")

**Ej. 2 — PyTorch equivalente.** El mismo loop: `zero_grad -> backward -> step`.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(60000, 784); Y = torch.randint(0, 10, (60000,))   # en real: FashionMNIST
dl = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)

model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.train()
for step, (xb, yb) in enumerate(dl):
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward()
    opt.step()
    if step % 200 == 0:
        print(f"step {step} | loss {loss.item():.3f}")
    if step == 400:
        break

**Ej. 3 — Lightning.** El mismo problema como `LightningModule`: el `Trainer` maneja el loop.

In [ ]:
import torch
import torch.nn as nn
import lightning as L

class LitMLP(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

    def training_step(self, batch, _):
        x, y = batch
        loss = nn.functional.cross_entropy(self.net(x), y)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

# L.Trainer(max_epochs=1).fit(LitMLP(), dl)   # sin zero_grad/backward/step manuales
print("Lightning: el training_step declara QUE hacer; el Trainer se ocupa del COMO")

**Ej. 4 — Speedup con jit.** `tf.function` en TF y `torch.compile` en PyTorch.

In [ ]:
import tensorflow as tf
import torch

@tf.function                       # compila el paso de entrenamiento a grafo
def train_step_tf(model, xb, yb, opt, loss_fn):
    with tf.GradientTape() as tape:
        loss = loss_fn(yb, model(xb, training=True))
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss

model = torch.nn.Linear(784, 10)
compiled = torch.compile(model)    # PyTorch 2.x: fusiona kernels via TorchInductor
print("tf.function y torch.compile: ~1.5-3x tras el warmup (JIT de kernels)")

**Ej. 5 — Multi-optimizer.** Un tape, dos optimizers: LR bajo al backbone, LR alto a la cabeza.

In [ ]:
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([keras.Input((784,)),
                          keras.layers.Dense(128, activation="relu", name="backbone"),
                          keras.layers.Dense(10, name="head")])
opt_slow = keras.optimizers.Adam(1e-5)   # capas preentrenadas: fine-tune suave
opt_fast = keras.optimizers.Adam(1e-3)   # cabeza nueva: aprende rapido
slow_vars = model.get_layer("backbone").trainable_variables
fast_vars = model.get_layer("head").trainable_variables
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        loss = loss_fn(yb, model(xb, training=True))
    g_slow, g_fast = tape.gradient(loss, [slow_vars, fast_vars])
    opt_slow.apply_gradients(zip(g_slow, slow_vars))
    opt_fast.apply_gradients(zip(g_fast, fast_vars))
    return loss

print("discriminative learning rates: trivial en loop manual, engorroso con model.fit()")